# EMA 6938 — Data Science for Materials
## Week 3 Take-Home Notebook: Probability, Statistics & Distributions

**Name:** *(your name here)*  
**Date:** *(date)*  
**Kernel:** Python (matds)

---

**Chapters:** Sandfeld Ch. 5–8  
**Format:** Fully take-home, no live lab this week  
**Assigned:** End of Day 1  
**Due:** Sunday 11:59 PM via Canvas

> **AI tool disclosure:** If you used any AI coding assistant (GitHub Copilot, ChatGPT, etc.) while completing this notebook, describe briefly which tool, for what purpose, and what you verified yourself. Delete this line if no AI tools were used.

---

### Files required
Place in the same directory as this notebook:
```
week03/
├── week3_statistics.ipynb        ← this file
└── data/
    └── week3_mp_bandgap_ef.csv   ← provided
```

### How to use this notebook
- **Demo cells** (`# LECTURE DEMO`) reproduce examples from the lectures. Run them, understand them, then extend them in task cells.
- **Task cells** (`# YOUR CODE HERE`) require you to write code.
- **Reflection cells** require written markdown answers — replace the italic placeholder text.
- Complete Parts A–C first (foundational). Parts D–F use real MP data and build on A–C.

---

In [ ]:
# Cell 0 — Environment check
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm, lognorm, poisson, expon, kstest
from scipy.stats import ttest_ind, f_oneway, kruskal

print(f"Python:   {sys.version.split()[0]}")
print(f"NumPy:    {np.__version__}")
print(f"pandas:   {pd.__version__}")
print(f"seaborn:  {sns.__version__}")
print("\n✓ All imports successful.")

# Set consistent plot style for the week
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'grid.linestyle': '--',
})

---
## Part A — Probability & Random Variables
**Connects to: Lecture 1, Sandfeld Ch. 5–6**

This part grounds the abstract definitions from the lecture in physical materials examples. Run each demo cell, make sure you understand the output, then complete the task cells.

### A1 — Poisson Vacancy Model
**Lecture demo — reproduce and understand**

The number of vacancies X in a crystal of N sites follows a Poisson distribution with parameter λ = N·exp(−Eₓ/kBT), where Eₓ is the vacancy formation energy.

In [ ]:
# Cell A1a — Poisson PMF: vacancy concentration in a crystal
# LECTURE DEMO

kB = 8.617e-5   # Boltzmann constant, eV/K
T  = 1500       # Temperature, K
Ef = 1.6        # Vacancy formation energy, eV (typical for Cu)
N  = 1_000_000  # Number of crystal sites

lambda_vac = N * np.exp(-Ef / (kB * T))
print(f"Expected number of vacancies (λ): {lambda_vac:.2f}")
print(f"Vacancy concentration: {lambda_vac/N:.2e}")

# PMF
k_max = int(lambda_vac * 3) + 1
k = np.arange(0, k_max)
pmf = poisson.pmf(k, lambda_vac)

fig, ax = plt.subplots(figsize=(8, 4))
# Gold bar on the mode, teal for all others
mode_k = int(lambda_vac)   # floor(λ) is the mode for Poisson
bar_colors = ['#F59E0B' if ki == mode_k else '#0D9488' for ki in k]
ax.bar(k, pmf, color=bar_colors, alpha=0.85, edgecolor='white', linewidth=0.6)
ax.axvline(lambda_vac, color='#EF4444', ls='--', lw=2, label=f'λ = E[X] = {lambda_vac:.1f}')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#F59E0B', label=f'Mode = floor(λ) = {mode_k}'),
    Patch(facecolor='#0D9488', label='P(X = k)'),
]
ax.legend(handles=legend_elements + ax.get_legend_handles_labels()[0], fontsize=11)

ax.set_xlabel('Number of vacancies (k)', fontsize=12)
ax.set_ylabel('P(X = k)', fontsize=12)
ax.set_title(f'Poisson Vacancy Distribution\n(Ef = {Ef} eV, T = {T} K, N = {N:,}, λ = {lambda_vac:.1f})',
             fontsize=12)

plt.tight_layout()
plt.savefig('A1_poisson_vacancies.png', dpi=150)
plt.show()

print(f"\nP(exactly 0 vacancies):  {poisson.pmf(0, lambda_vac):.4f}")
print(f"P(exactly λ vacancies):  {poisson.pmf(int(lambda_vac), lambda_vac):.4f}")
print(f"P(X ≤ λ):                {poisson.cdf(int(lambda_vac), lambda_vac):.4f}")

In [ ]:
# Cell A1b — Task: temperature dependence of vacancy concentration
# YOUR CODE HERE

# 1. Vary temperature from 500 K to 2000 K in 50 steps
#    Keep Ef = 1.6 eV, N = 1e6

# 2. For each temperature, compute λ (expected vacancies)
#    and the vacancy concentration c = λ/N

# 3. Plot: two-panel figure
#    Left: λ vs Temperature (K) — use log scale on y-axis
#    Right: ln(c) vs 1/T  (Arrhenius plot)
#    The slope of the Arrhenius plot should give you -Ef/kB

# 4. From the Arrhenius slope, extract Ef and compare to the input value

# YOUR CODE HERE


**A1b Reflection** *(answer in this cell)*

1. At 500 K, how many vacancies does your model predict? Is this physically reasonable for a real crystal?
2. The Arrhenius plot gives a straight line. What does this tell you about how Poisson λ depends on temperature?
3. Why is the Poisson distribution the right model for vacancies, and not Gaussian?

*Your answer here:*

### A2 — Gaussian Measurement Noise
**Lecture demo — reproduce and understand**

In [ ]:
# Cell A2a — Gaussian PDF and CDF
# LECTURE DEMO

mu    = 250.0   # True hardness (Vickers)
sigma = 15.0    # Measurement noise (std dev)

x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PDF
pdf_vals = norm.pdf(x, mu, sigma)
axes[0].plot(x, pdf_vals, color='#0D9488', lw=2.5)
for mult, alpha, label in [(1,0.30,'μ±σ (68.3%)'),(2,0.18,'μ±2σ (95.5%)'),(3,0.10,'μ±3σ (99.7%)')]:
    axes[0].fill_between(x, pdf_vals,
        where=((x >= mu - mult*sigma) & (x <= mu + mult*sigma)),
        alpha=alpha, color='#0D9488', label=label)
axes[0].set_xlabel('Hardness (HV)', fontsize=11)
axes[0].set_ylabel('Probability density', fontsize=11)
axes[0].set_title('PDF: Gaussian Measurement Noise', fontsize=12)
axes[0].legend(fontsize=10)

# CDF
axes[1].plot(x, norm.cdf(x, mu, sigma), color='#7C3AED', lw=2.5)
axes[1].axhline(0.5, color='gray', ls='--', alpha=0.6, label='50th percentile')
axes[1].axvline(mu, color='gray', ls='--', alpha=0.6)
axes[1].set_xlabel('Hardness (HV)', fontsize=11)
axes[1].set_ylabel('P(X ≤ x)', fontsize=11)
axes[1].set_title('CDF: Gaussian Measurement Noise', fontsize=12)
axes[1].legend(fontsize=10)

plt.suptitle(f'Gaussian Hardness Distribution: μ={mu}, σ={sigma} HV', fontsize=13)
plt.tight_layout()
plt.savefig('A2_gaussian_pdf_cdf.png', dpi=150)
plt.show()

# Compute a probability
p_above_270 = 1 - norm.cdf(270, mu, sigma)
print(f"P(hardness > 270 HV) = {p_above_270:.4f} = {p_above_270*100:.2f}%")

In [ ]:
# Cell A2b — Task: fitting a Gaussian to real hardness data
# YOUR CODE HERE

# Simulated hardness measurements (10 samples from a real material)
# In a real experiment, these would come from your instrument
np.random.seed(42)
hardness_data = np.random.normal(loc=248, scale=18, size=10)
print("Hardness measurements (HV):", np.round(hardness_data, 1))

# YOUR CODE HERE:
# 1. Compute the sample mean and sample standard deviation (ddof=1)


# 2. Compute the 95% confidence interval for the true mean
#    Formula: CI = x̄ ± t_{α/2, n-1} · s/√n
#    Use scipy.stats.t.ppf(0.975, df=n-1) for the t-critical value


# 3. Plot: histogram of the 10 measurements + the fitted Gaussian PDF
#    Mark the mean and 95% CI on the plot


# 4. Print: sample mean, sample std, 95% CI lower bound, 95% CI upper bound


**A2b Reflection** *(answer in this cell)*

1. Your sample mean is close to (but not equal to) 248. Why? What theorem guarantees it will get closer as n increases?
2. What does the 95% CI mean in plain language? Write a sentence a materials engineer would understand.
3. With only n=10 measurements, how wide is your CI? How many measurements would you need to halve the CI width? (Hint: CI width ∝ 1/√n)

*Your answer here:*

### A3 — Other Important Distributions
**Connecting to materials phenomena**

In [ ]:
# Cell A3 — Log-normal and Weibull distributions
# LECTURE DEMO

from scipy.stats import lognorm, weibull_min

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Log-normal: grain size distribution
x_grain = np.linspace(0.01, 30, 500)
sigma_ln, scale_ln = 0.6, 5.0   # shape and scale
axes[0].plot(x_grain, lognorm.pdf(x_grain, sigma_ln, scale=scale_ln),
             color='#0891B2', lw=2.5, label='Log-normal (grain size)')
axes[0].fill_between(x_grain, lognorm.pdf(x_grain, sigma_ln, scale=scale_ln),
                     alpha=0.2, color='#0891B2')
axes[0].set_xlabel('Grain diameter (μm)', fontsize=11)
axes[0].set_ylabel('Probability density', fontsize=11)
axes[0].set_title('Log-normal: Grain Size Distribution', fontsize=12)
axes[0].legend()

# Weibull: ceramic fracture strength
x_str = np.linspace(0, 600, 500)
for m, lbl in [(5, 'Weibull m=5 (brittle)'),
               (15, 'Weibull m=15 (ductile-like)')]:
    axes[1].plot(x_str, weibull_min.pdf(x_str, m, scale=300),
                 lw=2.5, label=lbl)
axes[1].set_xlabel('Fracture strength (MPa)', fontsize=11)
axes[1].set_ylabel('Probability density', fontsize=11)
axes[1].set_title('Weibull: Ceramic Fracture Strength', fontsize=12)
axes[1].legend(fontsize=10)

plt.suptitle('Materials-Relevant Distributions', fontsize=13)
plt.tight_layout()
plt.savefig('A3_materials_distributions.png', dpi=150)
plt.show()

print("Key Weibull insight:")
print("Higher Weibull modulus m → narrower distribution → more reliable ceramic.")
print("m=5: brittle, high scatter | m=20+: tight distribution, more predictable.")

---
## Part B — Distribution Fitting on Real MP Data
**Connects to: Lecture 3, Sandfeld Ch. 6**

This part uses the instructor-provided MP dataset. Load it first, then answer: *what distribution does each property follow, and why?*

In [ ]:
# Cell B1 — Load and inspect the MP dataset
# LECTURE DEMO

df = pd.read_csv('data/week3_mp_bandgap_ef.csv')

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nSummary statistics:")
print(df[['band_gap','Ef_eV_atom','volume_A3','density_g_cm3']].describe().round(3))
print(f"\nCrystal systems: {df['crystal_system'].value_counts().to_dict()}")
print(f"\nFraction metallic (gap=0): {(df['band_gap']==0).mean():.3f}")

In [ ]:
# Cell B2 — Bandgap distribution: full dataset
# LECTURE DEMO

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Full distribution
axes[0].hist(df['band_gap'].dropna(), bins=80, color='#1C2B4A', alpha=0.85,
             density=True, edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('Band gap (eV)', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title(f'All compounds\n(n={df.shape[0]:,})', fontsize=11)

# Non-metals only
df_nm = df[df['band_gap'] > 0.1]
axes[1].hist(df_nm['band_gap'], bins=60, color='#0D9488', alpha=0.85,
             density=True, edgecolor='white', linewidth=0.4)
axes[1].set_xlabel('Band gap (eV)', fontsize=11)
axes[1].set_title(f'Non-metals only (gap > 0.1 eV)\n(n={df_nm.shape[0]:,})', fontsize=11)

# Formation energy
axes[2].hist(df['Ef_eV_atom'].dropna(), bins=80, color='#7C3AED', alpha=0.85,
             density=True, edgecolor='white', linewidth=0.4)
axes[2].set_xlabel('Formation energy (eV/atom)', fontsize=11)
axes[2].set_title(f'Formation energy\n(n={df["Ef_eV_atom"].notna().sum():,})', fontsize=11)

fig.suptitle('MP Dataset — Raw Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('B2_raw_distributions.png', dpi=150)
plt.show()

In [ ]:
# Cell B3 — Fit a Gaussian to the non-metal bandgap distribution
# LECTURE DEMO

bg_nm = df[df['band_gap'] > 0.1]['band_gap'].dropna()

# Maximum likelihood fit
mu_fit, sig_fit = norm.fit(bg_nm)
print(f"Gaussian fit: μ = {mu_fit:.3f} eV, σ = {sig_fit:.3f} eV")

# KS test
ks_stat, ks_p = kstest(bg_nm, 'norm', args=(mu_fit, sig_fit))
print(f"KS test: statistic = {ks_stat:.4f}, p-value = {ks_p:.2e}")
print(f"Reject Gaussian hypothesis (α=0.05): {ks_p < 0.05}")

fig, ax = plt.subplots(figsize=(9, 5))
x = np.linspace(0, bg_nm.max(), 400)
ax.hist(bg_nm, bins=60, density=True, color='#0D9488', alpha=0.75,
        edgecolor='white', linewidth=0.4, label='Data')
ax.plot(x, norm.pdf(x, mu_fit, sig_fit), 'r-', lw=2.5,
        label=f'Gaussian fit\nμ={mu_fit:.2f}, σ={sig_fit:.2f} eV')
ax.set_xlabel('Band gap (eV)', fontsize=12)
ax.set_ylabel('Probability density', fontsize=12)
ax.set_title('Non-Metal Band Gap — Gaussian Fit & KS Test', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('B3_gaussian_fit_bandgap.png', dpi=150)
plt.show()

In [ ]:
# Cell B4 — Task: Gaussian fit to formation energy + comparison
# Do not use bg_nm df from Cell B3 here since it only contains bandgap for non-metals
# YOUR CODE HERE

ef_data = df['Ef_eV_atom'].dropna()

# 1. Fit a Gaussian to the formation energy distribution


# 2. Run a KS test


# 3. Plot: histogram + fitted Gaussian, with KS result in the title


# 4. Compare: does Ef_eV_atom fit a Gaussian better than band_gap?
#    Print both KS statistics and p-values side by side


**B4 Reflection** *(answer in this cell)*

1. Which property (band_gap or Ef_eV_atom) fits a Gaussian better? Report the KS statistics for both.
2. Propose a physical reason: why would formation energy be more Gaussian-like than bandgap?
   (Hint: think about how many independent physical mechanisms contribute to each property.)
3. The KS p-value is essentially 0 for both. Does this mean neither property is Gaussian? Explain the difference between statistical significance and practical significance here.

*Your answer here:*

In [ ]:
# Cell B5 — Task: Try a log-normal fit to the non-metal bandgap
# YOUR CODE HERE

# The Gaussian fit was rejected. Try log-normal:
# scipy.stats.lognorm — note: fit returns (shape, loc, scale)
# For a purely log-normal fit, fix loc=0: lognorm.fit(data, floc=0)

# 1. Fit log-normal to bg_nm


# 2. Run KS test for log-normal


# 3. Compute BIC for Gaussian and log-normal fits
#    BIC = k·ln(n) - 2·ln(L)
#    where k = number of parameters (2 for both), n = sample size,
#    L = maximised log-likelihood
#    Use: norm.logpdf(data, mu, sigma).sum()  and  lognorm.logpdf(data, ...).sum()


# 4. Plot: both fits overlaid on the histogram
#    Include BIC values in the legend


# 5. Which model wins by BIC? (Lower BIC = better fit)


---
## Part C — Expectation, Variance & the Central Limit Theorem
**Connects to: Lecture 2, Sandfeld Ch. 7**

In [ ]:
# Cell C1 — Compute mean and variance from first principles
# YOUR CODE HERE

# Use the MP dataset from Part B (df already loaded)
bg = df['band_gap'].dropna().values
ef = df['Ef_eV_atom'].dropna().values

# 1. Compute the sample mean WITHOUT using np.mean() or df.mean()
#    Use only np.sum() and len()
#    Then verify with np.mean()


# 2. Compute the sample variance WITHOUT using np.var() or df.var()
#    Remember: use ddof=1 (Bessel's correction — divide by n-1, not n)
#    Then verify with np.var(bg, ddof=1)


# 3. Print a summary table:
#    Property | n | Mean | Variance | Std Dev | Coefficient of Variation (CV = σ/μ × 100%)
#    for both band_gap and Ef_eV_atom


# 4. Interpret: which property has greater relative variability (higher CV)?
#    Does this match what you expected from the histograms in Part B?


In [ ]:
# Cell C2 — Law of Large Numbers demonstration
# LECTURE DEMO

# Running sample mean as n increases
np.random.seed(0)
bg_shuffled = df['band_gap'].dropna().values.copy()
np.random.shuffle(bg_shuffled)

true_mean = bg_shuffled.mean()
running_mean = np.cumsum(bg_shuffled) / np.arange(1, len(bg_shuffled)+1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(running_mean, color='#0D9488', lw=1.5, alpha=0.9, label='Running sample mean')
ax.axhline(true_mean, color='#EF4444', ls='--', lw=2, label=f'Population mean = {true_mean:.3f} eV')
ax.set_xlabel('Sample size n', fontsize=12)
ax.set_ylabel('Sample mean (eV)', fontsize=12)
ax.set_title('Law of Large Numbers — MP Band Gap', fontsize=12)
ax.set_xscale('log')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('C2_LLN.png', dpi=150)
plt.show()

# Quantify convergence
for n in [10, 100, 1000, 10000]:
    error = abs(running_mean[n-1] - true_mean)
    print(f"n={n:6d}: sample mean = {running_mean[n-1]:.4f} eV, |error| = {error:.4f} eV")

In [ ]:
# Cell C3 — Central Limit Theorem demonstration
# LECTURE DEMO

ef_data = df['Ef_eV_atom'].dropna().values
n_sizes = [1, 5, 20, 100]
n_simulations = 5000

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, n in zip(axes, n_sizes):
    sample_means = [np.mean(np.random.choice(ef_data, n)) for _ in range(n_simulations)]
    ax.hist(sample_means, bins=50, color='#7C3AED', alpha=0.8,
            density=True, edgecolor='white', linewidth=0.4)

    # Overlay theoretical Gaussian (CLT prediction)
    mu_clt = np.mean(ef_data)
    sig_clt = np.std(ef_data, ddof=1) / np.sqrt(n)
    x_range = np.linspace(min(sample_means), max(sample_means), 200)
    ax.plot(x_range, norm.pdf(x_range, mu_clt, sig_clt), 'r-', lw=2)

    ax.set_title(f'n = {n}\nσ_mean = {np.std(sample_means):.3f} eV\nTheory: {sig_clt:.3f} eV', fontsize=11)
    ax.set_xlabel('Sample mean Ef (eV/atom)', fontsize=10)

axes[0].set_ylabel('Density', fontsize=11)
fig.suptitle('Central Limit Theorem — Formation Energy\n(Red = CLT Gaussian prediction)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('C3_CLT.png', dpi=150)
plt.show()

In [ ]:
# Cell C4 — Task: 95% confidence interval for mean Ef using CLT
# YOUR CODE HERE

# 1. Compute the 95% confidence interval for the mean formation energy
#    Using the CLT formula: CI = x̄ ± z_{0.025} · σ/√n
#    z_{0.025} = 1.96 (from the standard normal table)
#    Use the full dataset (n = total number of non-null Ef values)


# 2. Now compute a t-interval (using the t-distribution instead of z)
#    Because we're estimating σ from data, the t-distribution is more correct:
#    CI = x̄ ± t_{0.025, df=n-1} · s/√n
#    Use: scipy.stats.t.ppf(0.975, df=n-1)


# 3. Print both CI bounds and the interval width for both approaches
#    Comment: why are they virtually identical for this n?


# 4. How large would n need to be so that the 95% CI width is ≤ 0.01 eV/atom?
#    Show the calculation.


---
## Part D — Correlation Matrix
**Connects to: Lecture 2, Sandfeld Ch. 7**

The correlation matrix is one of the first things you compute for any new dataset. It reveals redundant features and hidden structure-property relationships.

In [ ]:
# Cell D1 — Pearson and Spearman correlation matrices
# LECTURE DEMO

cols = ['band_gap', 'Ef_eV_atom', 'volume_A3', 'density_g_cm3']
df_num = df[cols].dropna()

corr_p = df_num.corr(method='pearson')
corr_s = df_num.corr(method='spearman')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for ax, corr, title in [
    (ax1, corr_p, 'Pearson (linear relationships)'),
    (ax2, corr_s, 'Spearman (rank-based, robust)'),
]:
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, square=True, ax=ax,
                linewidths=0.5, linecolor='white',
                cbar_kws={'label': 'Correlation coefficient', 'shrink': 0.8})
    ax.set_title(title, fontsize=12)

plt.suptitle('MP Dataset — Property Correlation Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('D1_correlation_matrices.png', dpi=150)
plt.show()

print("Pearson correlation matrix:")
print(corr_p.round(3))

In [ ]:
# Cell D2 — Task: identify and physically interpret key correlations
# YOUR CODE HERE

# 1. From the Pearson matrix, extract:
#    - The two most strongly correlated property pairs - positive and negative (highest ρ, excluding diagonal)
#    - The two most weakly correlated property pairs - positive and negative (lowest ρ)
#    Print them with their ρ values


# 2. For the strongest correlation:
#    Make a scatter plot of the two properties
#    Add a best-fit line (use np.polyfit(x, y, 1) or scipy.stats.linregress)
#    Label axes with units, add ρ value to the title


# 3. For the weakest correlation:
#    Make a scatter plot and verify visually that there is no clear trend


**D2 Reflection** *(answer in this cell)*

For each of your four identified property pairs, provide a physical explanation:

1. **Strongest positive correlation (ρ > 0):** What physical principle connects these two properties?
2. **Strongest negative correlation (ρ < 0):** Why do these properties move in opposite directions?
3. **Weakest correlation 1:** Why are these two properties essentially independent?
4. **Weakest correlation 2:** Same. What makes them independent?
5. Where do Pearson and Spearman disagree most? What does this tell you about the relationship between those properties?

*Your answer here:*

---
## Part E — Statistical Tests
**Connects to: Lecture 2, Sandfeld Ch. 8**

In [ ]:
# Cell E1 — Two-sample t-test: metals vs. insulators
# LECTURE DEMO

metals     = df[df['band_gap'] == 0.0]['Ef_eV_atom'].dropna()
insulators = df[df['band_gap'] > 1.0]['Ef_eV_atom'].dropna()

print(f"Metals:     n={len(metals):,}, mean={metals.mean():.3f} eV/atom, std={metals.std():.3f}")
print(f"Insulators: n={len(insulators):,}, mean={insulators.mean():.3f} eV/atom, std={insulators.std():.3f}")

t_stat, p_val = ttest_ind(metals, insulators, equal_var=False)  # Welch's t-test
print(f"\nWelch's t-test: t = {t_stat:.3f}, p = {p_val:.2e}")
print(f"Significant at α=0.05: {p_val < 0.05}")

# Effect size: Cohen's d
pooled_std = np.sqrt((metals.std()**2 + insulators.std()**2) / 2)
cohens_d = (metals.mean() - insulators.mean()) / pooled_std
print(f"Cohen's d = {cohens_d:.3f} (|d|>0.8: large effect)")

# Visualise
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(metals, bins=50, alpha=0.65, density=True, color='#1C2B4A',
        label=f'Metals (n={len(metals):,})')
ax.hist(insulators, bins=50, alpha=0.65, density=True, color='#0D9488',
        label=f'Insulators (n={len(insulators):,})')
ax.axvline(metals.mean(), color='#1C2B4A', ls='--', lw=2)
ax.axvline(insulators.mean(), color='#0D9488', ls='--', lw=2)
ax.set_xlabel('Formation energy (eV/atom)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'Metals vs. Insulators — Formation Energy\nt={t_stat:.2f}, p={p_val:.2e}, d={cohens_d:.2f}', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('E1_ttest.png', dpi=150)
plt.show()

In [ ]:
# Cell E2 — One-way ANOVA: bandgap across crystal systems
# LECTURE DEMO

systems = df['crystal_system'].value_counts()
systems = systems[systems >= 50].index.tolist()

groups = [(cs, df[df['crystal_system'] == cs]['band_gap'].dropna())
          for cs in systems]

labels = [cs for cs, _ in groups]
data   = [g.values for _, g in groups]

# Box plot first — always visualise before testing
fig, ax = plt.subplots(figsize=(11, 5))
bp = ax.boxplot(data,
                tick_labels=labels,
                patch_artist=True, notch=False,
                boxprops=dict(facecolor='#0D9488', alpha=0.7),
                medianprops=dict(color='white', lw=2),
                flierprops=dict(marker='.', markersize=2, alpha=0.3))
ax.set_xlabel('Crystal system', fontsize=12)
ax.set_ylabel('Band gap (eV)', fontsize=12)
ax.set_title('Band Gap Distribution by Crystal System', fontsize=12)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('E2_boxplot_crystal.png', dpi=150)
plt.show()

# ANOVA
f_stat, p_anova = f_oneway(*[g for _, g in groups])
print(f"One-way ANOVA: F = {f_stat:.2f}, p = {p_anova:.2e}")

# Kruskal-Wallis (non-parametric alternative)
h_stat, p_kw = kruskal(*[g for _, g in groups])
print(f"Kruskal-Wallis: H = {h_stat:.2f}, p = {p_kw:.2e}")

print(f"\nMedian bandgap by crystal system:")
for cs, g in sorted(groups, key=lambda x: x[1].median(), reverse=True):
    print(f"  {cs:15s}: median = {g.median():.3f} eV, n = {len(g):,}")

In [ ]:
# Cell E3 — Task: pairwise t-test with effect size
# YOUR CODE HERE

# Choose any two crystal systems from the list above
CS1 = 'Cubic'
CS2 = 'Hexagonal'

group1 = df[df['crystal_system'] == CS1]['band_gap'].dropna()
group2 = df[df['crystal_system'] == CS2]['band_gap'].dropna()

# 1. Print descriptive statistics for each group (n, mean, median, std)


# 2. Run Welch's two-sample t-test (equal_var=False)


# 3. Compute Cohen's d (effect size)
#    Cohen's d = |mean1 - mean2| / pooled_std
#    Convention: d < 0.2 small, 0.2–0.8 medium, > 0.8 large


# 4. Print: t-statistic, p-value, Cohen's d, and your interpretation


**E3 Reflection** *(answer in this cell)*

1. Is the difference between your two crystal systems statistically significant? (Report t, p, and α)
2. What is the effect size (Cohen's d), and how do you interpret it?
3. In one sentence: what is the physical reason that cubic and hexagonal compounds might have different average bandgaps?
4. **Critical thinking:** if p < 0.05 but Cohen's d = 0.05 (small effect), should you include crystal system as a feature in an ML model? Why or why not?

*Your answer here:*

---
## Part F — Reflection
**Take-home · Connects all lectures**

### F1 — Connecting statistics to ML

In 3–4 sentences, explain how at least two statistical concepts from this week underpin ML methods we will use in Weeks 4–8.

Be specific: name the statistical concept, the ML method it connects to, and the mechanism. For example: *"Standardisation (subtracting E[X] and dividing by Var(X)^0.5) is applied before PCA and regularised regression because these methods are sensitive to the scale of features without standardisation, large-magnitude features dominate the model regardless of their information content."*

Choose different concepts from the example above.

*Your answer here (3–4 sentences, ≥ 2 connections):*

### F2 — Your own research/research interests/occupation

Think about a property you measure or compute in your own research/research interests/occupation.

1. **What distribution would you expect it to follow?** Name the distribution (Gaussian, log-normal, Weibull, Poisson, etc.) and give the physical reason for your choice and not just 'it's common'.

2. **How would you test this?** Describe the statistical test you would use (KS test, Shapiro-Wilk, Q-Q plot, etc.) and what sample size you would need.

3. **What would a non-Gaussian distribution imply for your data analysis?** For example: if you use the sample mean to characterise the property, what does the CLT tell you about when that estimate becomes reliable?

*Your answer here (3 parts, research-specific and not generic):*

---
## Submission Checklist

Before submitting via Canvas (due **Sunday 11:59 PM**):

**Part A — Probability & Distributions**
- [ ] A1a: Poisson plot reproduced and annotated
- [ ] A1b: Temperature dependence + Arrhenius plot completed, Ef extracted from slope
- [ ] A1b Reflection: answered
- [ ] A2a: Gaussian PDF/CDF reproduced, P(hardness > 270 HV) computed
- [ ] A2b: Gaussian fit to hardness data, 95% CI computed and plotted
- [ ] A2b Reflection: answered
- [ ] A3: Log-normal and Weibull plots reproduced

**Part B — Distribution Fitting**
- [ ] B1: MP dataset loaded, shape/missing/summary printed
- [ ] B2: Three histograms produced
- [ ] B3: Gaussian fit to non-metal bandgap, KS test run
- [ ] B4: Gaussian fit to Ef, comparison to B3, reflection answered
- [ ] B5: Log-normal fit, BIC comparison, better model identified

**Part C — Expectation, Variance, CLT**
- [ ] C1: Mean and variance computed from first principles, CV reported
- [ ] C2: LLN running mean plot reproduced
- [ ] C3: CLT demonstration with 4 sample sizes reproduced
- [ ] C4: 95% CI computed (z and t methods), n for CI width ≤ 0.01 calculated

**Part D — Correlation Matrix**
- [ ] D1: Pearson and Spearman matrices computed and plotted
- [ ] D2: Strongest/weakest pairs identified, scatter plots made, reflection answered

**Part E — Statistical Tests**
- [ ] E1: t-test metals vs. insulators reproduced, Cohen's d computed
- [ ] E2: ANOVA and Kruskal-Wallis reproduced, box plot produced
- [ ] E3: Pairwise t-test for chosen crystal systems, effect size, reflection answered

**Part F — Reflection**
- [ ] F1: 3–4 sentences connecting Week 3 statistics to ML (specific, not generic)
- [ ] F2: Your research property, distribution, test, implications

**Final steps**
- [ ] **Kernel → Restart & Run All** All cells execute without errors
- [ ] All plots display inline
- [ ] All reflection cells filled in (no placeholder text remaining)
- [ ] Save notebook before uploading
- [ ] AI disclosure note updated or deleted at the top of the notebook
- [ ] File renamed: `[LastName]_week3.ipynb`